### Import libraries

In [ ]:
! pip install snntorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 3.2 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import snntorch as snn
from snntorch import surrogate

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import os
import time

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, epoch, device):
    """Train for one epoch."""
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()

        output = model(images)

        # Handle models that return (logits, spike_probs) vs just logits
        if isinstance(output, tuple):
            logits, spike_probs = output
        else:
            logits = output

        # Compute loss
        loss = criterion(logits, labels)

        # Backward pass
        loss.backward()

        # Gradient clipping (helps stability with surrogate gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Track statistics
        running_loss += loss.item()
        _, predicted = logits.max(dim=1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Print progress
        if (batch_idx + 1) % 100 == 0:
            avg_loss = running_loss / (batch_idx + 1)
            accuracy = 100.0 * correct / total
            print(f"  Batch {batch_idx + 1}/{len(train_loader)}: "
                  f"Loss = {avg_loss:.4f}, Acc = {accuracy:.2f}%")

    epoch_time = time.time() - start_time
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc, epoch_time


def evaluate(model, test_loader, criterion, device):
    """Evaluate model on test set."""
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    # Track spike statistics (if available)
    all_spike_probs = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            # Get model output
            output = model(images)

            # Handle models that return (logits, spike_probs) vs just logits
            if isinstance(output, tuple):
                logits, spike_probs = output
                if spike_probs is not None:
                    all_spike_probs.append(spike_probs.cpu())
            else:
                logits = output

            loss = criterion(logits, labels)

            running_loss += loss.item()
            _, predicted = logits.max(dim=1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    # Compute statistics
    test_loss = running_loss / len(test_loader)
    test_acc = 100.0 * correct / total

    # Spike probability statistics (if available)
    if all_spike_probs:
        all_spike_probs = torch.cat(all_spike_probs, dim=0)
        mean_spike_prob = all_spike_probs.mean().item()
        std_spike_prob = all_spike_probs.std().item()
    else:
        mean_spike_prob = None
        std_spike_prob = None

    return test_loss, test_acc, mean_spike_prob, std_spike_prob

In [ ]:
class FullySpikingCNN(nn.Module):

    def __init__(self, num_timesteps=20, num_classes=10, beta=0.95):
        super().__init__()
        self.num_timesteps = num_timesteps

        self.w_max = 0.1
        self.raw_w = nn.Parameter(torch.zeros(3))

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.lif_c1 = snn.Leaky(beta=beta, spike_grad=surrogate.fast_sigmoid())

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.lif_c2 = snn.Leaky(beta=beta, spike_grad=surrogate.fast_sigmoid())

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.lif_c3 = snn.Leaky(beta=beta, spike_grad=surrogate.fast_sigmoid())

        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.lif_f1 = snn.Leaky(beta=beta, spike_grad=surrogate.fast_sigmoid())

        self.fc2 = nn.Linear(512, 256)
        self.lif_f2 = snn.Leaky(beta=beta, spike_grad=surrogate.fast_sigmoid())

        self.fc_out = nn.Linear(256, num_classes)

    def _input_spikes(self, x, t):

        w = (self.w_max * torch.tanh(self.raw_w)).view(1, 3, 1, 1)

        rates = x * torch.exp(-w * t)
        probs = torch.clamp(rates, 0.0, 1.0)

        sampled = torch.bernoulli(probs)
        return probs + (sampled - probs).detach()

    def forward(self, x):

        mem_c1 = self.lif_c1.init_leaky()
        mem_c2 = self.lif_c2.init_leaky()
        mem_c3 = self.lif_c3.init_leaky()
        mem_f1 = self.lif_f1.init_leaky()
        mem_f2 = self.lif_f2.init_leaky()

        logits_over_time = []

        for t in range(self.num_timesteps):
            spk = self._input_spikes(x, t)

            cur = self.conv1(spk)
            spk, mem_c1 = self.lif_c1(cur, mem_c1)
            spk = F.max_pool2d(spk, 2)

            cur = self.conv2(spk)
            spk, mem_c2 = self.lif_c2(cur, mem_c2)
            spk = F.max_pool2d(spk, 2)

            cur = self.conv3(spk)
            spk, mem_c3 = self.lif_c3(cur, mem_c3)
            spk = F.max_pool2d(spk, 2)

            cur = self.fc1(spk.flatten(1))
            spk, mem_f1 = self.lif_f1(cur, mem_f1)

            cur = self.fc2(spk)
            spk, mem_f2 = self.lif_f2(cur, mem_f2)

            logits_over_time.append(self.fc_out(spk))
        return torch.stack(logits_over_time, dim=0).sum(dim=0)

    def learned_w(self):
        return (self.w_max * torch.tanh(self.raw_w)).detach().cpu().tolist()


def get_fully_spiking_data_loaders(batch_size=128, num_workers=2):

    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor()
    ])

    test_transform = transforms.Compose([
        transforms.ToTensor()
    ])

    pin_memory = (device.type == "cuda")

    train_loader = DataLoader(
        torchvision.datasets.CIFAR10(
            root='./data', train=True, download=True, transform=train_transform
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

    test_loader = DataLoader(
        torchvision.datasets.CIFAR10(
            root='./data', train=False, download=True, transform=test_transform
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

    return train_loader, test_loader


def train_fully_spiking(
    model,
    model_name="fully_spiking_cnn",
    num_epochs=100,
    batch_size=128,
    learning_rate=1e-3,
    save_dir='./checkpoints'
):

    print("=" * 60)
    print(f"Training: {model_name}")
    print("=" * 60)

    os.makedirs(save_dir, exist_ok=True)

    print("\nLoading data...")
    train_loader, test_loader = get_fully_spiking_data_loaders(
        batch_size=batch_size,
        num_workers=2
    )
    print(f"Training samples: {len(train_loader.dataset)}")
    print(f"Test samples: {len(test_loader.dataset)}")

    print("\nInitializing model...")
    model = model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=1e-6
    )

    history = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': [],
        'learning_rate': [],
        'w_r': [],
        'w_g': [],
        'w_b': []
    }

    best_test_acc = 0.0

    print("\nStarting training...")
    print("-" * 60)

    for epoch in range(num_epochs):
        current_lr = optimizer.param_groups[0]['lr']
        print(f"\nEpoch {epoch + 1}/{num_epochs} (LR: {current_lr:.6f})")

        train_loss, train_acc, epoch_time = train_one_epoch(
            model, train_loader, criterion, optimizer, epoch, device
        )

        test_loss, test_acc, _, _ = evaluate(
            model, test_loader, criterion, device
        )

        scheduler.step()

        w_r, w_g, w_b = model.learned_w()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['learning_rate'].append(current_lr)
        history['w_r'].append(w_r)
        history['w_g'].append(w_g)
        history['w_b'].append(w_b)

        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Test Loss:  {test_loss:.4f}, Test Acc:  {test_acc:.2f}%")
        print(f"  Temporal w RGB: [{w_r:.4f}, {w_g:.4f}, {w_b:.4f}]")
        print(f"  Time: {epoch_time:.1f}s")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'test_acc': test_acc,
                'history': history
            }, os.path.join(save_dir, f'best_{model_name}.pt'))
            print(f"  *** New best model saved (Acc: {test_acc:.2f}%) ***")
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'history': history
            }, os.path.join(save_dir, f'{model_name}_epoch_{epoch + 1}.pt'))


    print("\n" + "=" * 60)
    print("Training complete!")
    print(f"Best test accuracy: {best_test_acc:.2f}%")
    print("=" * 60)

    return model, history


In [ ]:
fully_spiking_model = FullySpikingCNN(
    num_timesteps=20,
    num_classes=10,
    beta=0.95
)

fully_spiking_model, fully_spiking_history = train_fully_spiking(
    model=fully_spiking_model,
    model_name="fully_spiking_cnn",
    num_epochs=100,
    batch_size=128,
    learning_rate=1e-3,
    save_dir='./checkpoints'
)


Training: fully_spiking_cnn

Loading data...


100%|██████████| 170M/170M [00:02<00:00, 78.5MB/s]


Training samples: 50000
Test samples: 10000

Initializing model...
Total parameters: 1,276,237
Trainable parameters: 1,276,237

Starting training...
------------------------------------------------------------

Epoch 1/100 (LR: 0.001000)
  Batch 100/391: Loss = 2.1447, Acc = 19.98%
  Batch 200/391: Loss = 1.9932, Acc = 25.70%
  Batch 300/391: Loss = 1.9074, Acc = 29.03%
  Train Loss: 1.8516, Train Acc: 31.12%
  Test Loss:  1.5835, Test Acc:  41.03%
  Temporal w RGB: [0.0002, 0.0012, -0.0007]
  Time: 82.1s
  *** New best model saved (Acc: 41.03%) ***

Epoch 2/100 (LR: 0.001000)
  Batch 100/391: Loss = 1.6213, Acc = 40.12%
  Batch 200/391: Loss = 1.5981, Acc = 41.24%
  Batch 300/391: Loss = 1.5776, Acc = 42.10%
  Train Loss: 1.5595, Train Acc: 42.67%
  Test Loss:  1.3768, Test Acc:  49.30%
  Temporal w RGB: [-0.0013, 0.0002, -0.0006]
  Time: 82.8s
  *** New best model saved (Acc: 49.30%) ***

Epoch 3/100 (LR: 0.000999)
  Batch 100/391: Loss = 1.4658, Acc = 46.06%
  Batch 200/391: Loss = 